In [1]:
import pandas as pd

# OWID Energy dataset
url = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
df = pd.read_csv(url)

# Keep relevant columns
cols = [
    "country", "year",
    "electricity_generation",
    "electricity_demand",
    "renewables_consumption",
    "fossil_fuel_consumption",
    "primary_energy_consumption"
]

df = df[cols]

# Filter recent years
df = df[df["year"] >= 2010]

# Drop missing
df = df.dropna()

df.head()

,country,year,electricity_generation,electricity_demand,renewables_consumption,fossil_fuel_consumption,primary_energy_consumption
261,Africa,2010,672.41,676.41,299.146,4106.646,4460.190
262,Africa,2011,688.68,694.88,307.544,4169.005,4529.279
263,Africa,2012,725.45,733.60,309.049,4372.651,4734.076
264,Africa,2013,740.91,750.30,326.263,4505.302,4888.162
265,Africa,2014,768.06,778.00,351.851,4641.549,5049.482


In [3]:
df.to_csv('owid_energy_data_processed.csv', index=False)
print("Data saved to 'owid_energy_data_processed.csv'")

Data saved to 'owid_energy_data_processed.csv'


In [4]:
#columns selection
cols = [
    "country", "year",
    "electricity_generation",
    "electricity_demand",
    "renewables_consumption",
    "fossil_fuel_consumption",
    "primary_energy_consumption"
]

We don't need the entire dataset (it has too many columns).
So, we select only:

country
year
electricity generation
consumption
renewables
fossil fuels

In [5]:
#filter data (2010+)
df = df[df["year"] >= 2010]

In [6]:
df.to_csv('df_2010+.csv', index=False)
print("Data saved to 'df_2010+.csv'")

Data saved to 'df_2010+.csv'


We retain only recent data (2010 onwards).

Why:
more relevant to today's market
more stable energy market structure
we avoid outdated economic regimes
“I filtered the dataset to recent years (post-2010) to ensure structural consistency in energy markets

In [8]:
#DROP MISSING VALUES
df = df.dropna()
df.head()

,country,year,electricity_generation,electricity_demand,renewables_consumption,fossil_fuel_consumption,primary_energy_consumption
261,Africa,2010,672.41,676.41,299.146,4106.646,4460.190
262,Africa,2011,688.68,694.88,307.544,4169.005,4529.279
263,Africa,2012,725.45,733.60,309.049,4372.651,4734.076
264,Africa,2013,740.91,750.30,326.263,4505.302,4888.162
265,Africa,2014,768.06,778.00,351.851,4641.549,5049.482


EU FILTERING (crucial for the portfolio)
  we aim to achieve:To retain only the following from the global dataset:

🇪🇺 European countries (EU + countries relevant to the European energy market)

The dataset is:

global
mixed markets (US, Asia, Africa)
 We convert it to:

“European-focused energy market dataset for regional analysis”

In [9]:
#Create EU country list
eu_countries = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus",
    "Czechia", "Denmark", "Estonia", "Finland", "France",
    "Germany", "Greece", "Hungary", "Ireland", "Italy",
    "Latvia", "Lithuania", "Luxembourg", "Malta",
    "Netherlands", "Poland", "Portugal", "Romania",
    "Slovakia", "Slovenia", "Spain", "Sweden"
]

In [10]:
df_eu = df[df["country"].isin(eu_countries)]

In [11]:
df_eu["country"].unique()

array(['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia',
       'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece',
       'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg',
       'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia',
       'Slovenia', 'Spain', 'Sweden'], dtype=object)

In [12]:
#clean up before powerbi
df_eu = df_eu.drop_duplicates()
df_eu = df_eu.reset_index(drop=True)

KPI ENGINEERING


In [13]:
#Renewable Share %
df_eu["renewable_share"] = (
    df_eu["renewables_consumption"] /
    df_eu["primary_energy_consumption"]
) * 100

I created a renewable energy penetration KPI by calculating the share of renewables over total primary energy consumption.

In [14]:
#Fossil Dependency %
df_eu["fossil_dependency"] = (
    df_eu["fossil_fuel_consumption"] /
    df_eu["primary_energy_consumption"]
) * 100

I calculated fossil fuel dependency to measure structural reliance on non-renewable energy sources across EU countries.

In [15]:
#Energy Intensity Proxy
df_eu["energy_intensity_proxy"] = (
    df_eu["primary_energy_consumption"] /
    df_eu["electricity_generation"]
)

I created an energy intensity proxy to compare efficiency of electricity generation across countries.

In [16]:
#Year-over-Year Change (trend signal)
df_eu["consumption_yoy_change"] = df_eu.groupby("country")[
    "primary_energy_consumption"
].pct_change() * 100

I computed year-over-year changes in energy consumption using group-based time series transformations.

In [17]:
#Renewable Transition Gap
df_eu["transition_gap"] = df_eu["fossil_dependency"] - df_eu["renewable_share"]

I created a composite KPI called energy transition gap to quantify the imbalance between fossil dependency and renewable penetration.

POWER BI DATA MODEL

The fact table contains energy metrics and engineered KPIs at country-year level, enabling time-series and cross-country analysis.

In [18]:
#DIMENSION — dim_date
dim_date = pd.DataFrame({
    "year": sorted(df_eu["year"].unique())
})

created a dedicated date dimension to enable time intelligence functions such as year-over-year analysis.